# 12 · Préparation des données — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch12_preparation/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch12_preparation/03_notebook.ipynb`, créée par `python tools/start_chapter.py 12`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 12.Q1 à 12.Q11, 12.R1 à 12.R3, 12.1 à 12.5, 12.7 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★★ | |
| A | 12.11 à 12.15 | diagnostiquer Penguins brut, un CSV à la française, où tombent les données de test après un MinMaxScaler, dates, jointures et tableaux croisés avec pandas, une base SQL (*prochaine session de génération*) | | | |
| B | 12.16 à 12.21 | StandardScaler, MinMaxScaler et SimpleImputer dans mylearn.preprocessing, scalers face aux points aberrants, bugs de préparation, transformer la cible (*prochaine session de génération*) | | | |
| C | 12.22 à 12.28 | projeter un nuage, variance expliquée, MNIST en composantes, sélection de features, OrdinalEncoder, OneHotEncoder et PCA dans mylearn.preprocessing (*prochaine session de génération*) | | | |
| D | 12.29 à 12.33 | échelle et descente de gradient, PCA, t-SNE et UMAP, fuites par le prétraitement, une préparation testée, défi Penguins brut (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="12")  # your own library

## Objectifs et rappel express

- Diagnostiquer et nettoyer un dataset brut avec pandas, et l'assembler avec des jointures et SQL.
- Écrire et vérifier contre scikit-learn les transformateurs de `mylearn.preprocessing` : `StandardScaler`, `MinMaxScaler`, `SimpleImputer`, `OrdinalEncoder`, `OneHotEncoder` et `PCA`.
- Choisir une mise à l'échelle et un encodage, et appliquer la règle d'or : `fit` sur l'entraînement, `transform` partout.
- Réduire et visualiser des données en grande dimension (PCA, t-SNE, UMAP), et mesurer les fuites dues au prétraitement.

**Rappel express.** Min-max : $x' = a + (b - a)\,\frac{x - \min}{\max - \min}$ ; standardisation : $z = \frac{x - \mu}{\sigma}$ (ddof = 0), inverse $x = z\,\sigma + \mu$ ; covariance : $\boldsymbol{\Sigma} = \frac{1}{n - 1}\mathbf{X}_c^\top\mathbf{X}_c$ ; variance des projections sur $\mathbf{u}$ unitaire : $\mathbf{u}^\top\boldsymbol{\Sigma}\,\mathbf{u}$ ; PCA : $\mathbf{Z} = (\mathbf{X} - \boldsymbol{\mu})\,\mathbf{W}_k^\top$, $\hat{\mathbf{X}} = \mathbf{Z}\,\mathbf{W}_k + \boldsymbol{\mu}$, part de variance expliquée $\lambda_j / \sum_l \lambda_l$. En Python : `sklearn.preprocessing` (`StandardScaler`, `MinMaxScaler`, `RobustScaler`, `OrdinalEncoder`, `OneHotEncoder`), `sklearn.impute.SimpleImputer`, `sklearn.decomposition.PCA`, `sklearn.pipeline.make_pipeline`, `np.linalg.svd`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 12.1 à 12.5 et 12.7)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres collées (`"BF"`) ; une suite de codes : les lettres dans l'ordre (`"XYZ"`) ; plusieurs nombres : une liste (`[4, 7]`) ; une matrice : une liste de lignes (`[[1, 1], [1, 1]]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 12.6 et 12.8, la réflexion (🗣️ 12.9, ⚖️ 12.10) et l'entretien se corrigent avec `05_solutions.md`.

**12.Q1 — La règle d'or de la préparation**

In [ ]:
# 12.Q1: the values you found on paper
answer_12_Q1a = ...  # a) the letter of your choice, e.g. "E"
answer_12_Q1b = ...  # b) the letter of your choice
answer_12_Q1c = ...  # c) True or False
answer_12_Q1d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_12_Q1a, answer_12_Q1b, answer_12_Q1c, answer_12_Q1d]):
    wb.check(f"12.Q1{letter}", answer)

**12.Q2 — Numérique, ordinale ou nominale ?**

In [ ]:
# 12.Q2: the values you found on paper
answer_12_Q2a = ...  # a) six letters Q, O or N, in the order of the items, e.g. "XYZXYZ"
answer_12_Q2b = ...  # b) the letter of your choice
answer_12_Q2c = ...  # c) True or False

for letter, answer in zip("abc", [answer_12_Q2a, answer_12_Q2b, answer_12_Q2c]):
    wb.check(f"12.Q2{letter}", answer)

**12.Q3 — Pourquoi un one-hot plutôt qu'un entier ?**

In [ ]:
# 12.Q3: the values you found on paper
answer_12_Q3a = ...  # a) the letter of your choice
answer_12_Q3b = ...  # b) a whole number
answer_12_Q3c = ...  # c) a whole number
answer_12_Q3d = ...  # d) a list of 0s and 1s, e.g. [1, 1, 1]
answer_12_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_12_Q3a, answer_12_Q3b, answer_12_Q3c, answer_12_Q3d, answer_12_Q3e]):
    wb.check(f"12.Q3{letter}", answer)

**12.Q4 — Doublons, NaN et points aberrants**

In [ ]:
# 12.Q4: the values you found on paper
answer_12_Q4a = ...  # a) the letter of your choice
answer_12_Q4b = ...  # b) True or False
answer_12_Q4c = ...  # c) True or False
answer_12_Q4d = ...  # d) the letter of your choice
answer_12_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_12_Q4a, answer_12_Q4b, answer_12_Q4c, answer_12_Q4d, answer_12_Q4e]):
    wb.check(f"12.Q4{letter}", answer)

**12.Q5 — Normaliser ou standardiser ?**

In [ ]:
# 12.Q5: the values you found on paper
answer_12_Q5a = ...  # a) the letter of your choice
answer_12_Q5b = ...  # b) True or False
answer_12_Q5c = ...  # c) True or False
answer_12_Q5d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_12_Q5a, answer_12_Q5b, answer_12_Q5c, answer_12_Q5d]):
    wb.check(f"12.Q5{letter}", answer)

**12.Q6 — Données de test hors de [0, 1] : bug ou normal ?**

In [ ]:
# 12.Q6: the values you found on paper
answer_12_Q6a = ...  # a) a number with 2 decimals
answer_12_Q6b = ...  # b) True or False
answer_12_Q6c = ...  # c) a number
answer_12_Q6d = ...  # d) the letter of your choice

for letter, answer in zip("abcd", [answer_12_Q6a, answer_12_Q6b, answer_12_Q6c, answer_12_Q6d]):
    wb.check(f"12.Q6{letter}", answer)

**12.Q7 — Univarié ou multivarié ?**

In [ ]:
# 12.Q7: the values you found on paper
answer_12_Q7a = ...  # a) six letters U or M, in the order of the items
answer_12_Q7b = ...  # b) True or False

for letter, answer in zip("ab", [answer_12_Q7a, answer_12_Q7b]):
    wb.check(f"12.Q7{letter}", answer)

**12.Q8 — Sélectionner ou réduire la dimension ?**

In [ ]:
# 12.Q8: the values you found on paper
answer_12_Q8a = ...  # a) five letters S or R, in the order of the items
answer_12_Q8b = ...  # b) True or False
answer_12_Q8c = ...  # c) True or False

for letter, answer in zip("abc", [answer_12_Q8a, answer_12_Q8b, answer_12_Q8c]):
    wb.check(f"12.Q8{letter}", answer)

**12.Q9 — Ce que fait (et ne fait pas) une PCA**

In [ ]:
# 12.Q9: the values you found on paper
answer_12_Q9a = ...  # a) True or False
answer_12_Q9b = ...  # b) True or False
answer_12_Q9c = ...  # c) True or False
answer_12_Q9d = ...  # d) True or False
answer_12_Q9e = ...  # e) True or False
answer_12_Q9f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_12_Q9a, answer_12_Q9b, answer_12_Q9c, answer_12_Q9d, answer_12_Q9e, answer_12_Q9f]):
    wb.check(f"12.Q9{letter}", answer)

**12.Q10 — Quelle découpe pour ces données ?**

In [ ]:
# 12.Q10: the values you found on paper
answer_12_Q10a = ...  # a) six letters L, C or E, in the order of the items
answer_12_Q10b = ...  # b) True or False

for letter, answer in zip("ab", [answer_12_Q10a, answer_12_Q10b]):
    wb.check(f"12.Q10{letter}", answer)

**12.Q11 — Où se cache la fuite ?**

In [ ]:
# 12.Q11: the values you found on paper
answer_12_Q11a = ...  # a) the letters, in alphabetical order, e.g. "BF"
answer_12_Q11b = ...  # b) True or False

for letter, answer in zip("ab", [answer_12_Q11a, answer_12_Q11b]):
    wb.check(f"12.Q11{letter}", answer)

**12.R1 — Représentation, évaluation, optimisation — où placer la préparation ?**

In [ ]:
# 12.R1: the values you found on paper
answer_12_R1a = ...  # a) the letter of your choice
answer_12_R1b = ...  # b) the letter of your choice
answer_12_R1c = ...  # c) the letter of your choice
answer_12_R1d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_12_R1a, answer_12_R1b, answer_12_R1c, answer_12_R1d]):
    wb.check(f"12.R1{letter}", answer)

**12.R2 — Pourquoi la pénalité ridge dépend de l'échelle des features**

In [ ]:
# 12.R2: the values you found on paper
answer_12_R2a = ...  # a) the letter of your choice
answer_12_R2b = ...  # b) a whole number
answer_12_R2c = ...  # c) the letter of your choice
answer_12_R2d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_12_R2a, answer_12_R2b, answer_12_R2c, answer_12_R2d]):
    wb.check(f"12.R2{letter}", answer)

**12.R3 — Descente de gradient dans une vallée très allongée**

In [ ]:
# 12.R3: the values you found on paper
answer_12_R3a = ...  # a) a list of two numbers
answer_12_R3b = ...  # b) a number
answer_12_R3c = ...  # c) a number
answer_12_R3d = ...  # d) a number
answer_12_R3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_12_R3a, answer_12_R3b, answer_12_R3c, answer_12_R3d, answer_12_R3e]):
    wb.check(f"12.R3{letter}", answer)

**Ex 12.1 — One-hot à la main sur Penguins**

In [ ]:
# Ex 12.1: the values you found on paper
answer_12_1a = ...  # a) a whole number
answer_12_1b = ...  # b) a list of numbers
answer_12_1c = ...  # c) a whole number
answer_12_1d = ...  # d) a list of numbers
answer_12_1e = ...  # e) a whole number
answer_12_1f = ...  # f) a list of numbers
answer_12_1g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_12_1a, answer_12_1b, answer_12_1c, answer_12_1d, answer_12_1e, answer_12_1f, answer_12_1g]):
    wb.check(f"12.1{letter}", answer)

**Ex 12.2 — Min-max et z-score de cinq valeurs**

In [ ]:
# Ex 12.2: the values you found on paper
answer_12_2a = ...  # a) a number
answer_12_2b = ...  # b) a number
answer_12_2c = ...  # c) a number with 3 decimals
answer_12_2d = ...  # d) a number
answer_12_2e = ...  # e) a number
answer_12_2f = ...  # f) a number with 3 decimals
answer_12_2g = ...  # g) a number with 3 decimals

for letter, answer in zip("abcdefg", [answer_12_2a, answer_12_2b, answer_12_2c, answer_12_2d, answer_12_2e, answer_12_2f, answer_12_2g]):
    wb.check(f"12.2{letter}", answer)

**Ex 12.3 — Mise à l'échelle univariée ou multivariée**

In [ ]:
# Ex 12.3: the values you found on paper
answer_12_3a = ...  # a) a list of 3 numbers
answer_12_3b = ...  # b) a list of 3 numbers
answer_12_3c = ...  # c) a number
answer_12_3d = ...  # d) True or False
answer_12_3e = ...  # e) the letter of your choice

for letter, answer in zip("abcde", [answer_12_3a, answer_12_3b, answer_12_3c, answer_12_3d, answer_12_3e]):
    wb.check(f"12.3{letter}", answer)

**Ex 12.4 — Réappliquer la transformation : −10 °C, −50 °C et retour aux voitures**

In [ ]:
# Ex 12.4: the values you found on paper
answer_12_4a = ...  # a) a number with 3 decimals
answer_12_4b = ...  # b) a number of cars
answer_12_4c = ...  # c) a number with 3 decimals
answer_12_4d = ...  # d) a number with 3 decimals
answer_12_4e = ...  # e) True or False
answer_12_4f = ...  # f) a number of cars
answer_12_4g = ...  # g) a number with 3 decimals
answer_12_4h = ...  # h) a number of cars
answer_12_4i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_12_4a, answer_12_4b, answer_12_4c, answer_12_4d, answer_12_4e, answer_12_4f, answer_12_4g, answer_12_4h, answer_12_4i]):
    wb.check(f"12.4{letter}", answer)

**Ex 12.5 — Trois découpes d'un même tableau : échantillon, feature, élément**

In [ ]:
# Ex 12.5: the values you found on paper
answer_12_5a = ...  # a) a list of 3 numbers
answer_12_5b = ...  # b) a list of 3 numbers
answer_12_5c = ...  # c) a number with 3 decimals
answer_12_5d = ...  # d) a list of 3 numbers
answer_12_5e = ...  # e) the letters, in alphabetical order
answer_12_5f = ...  # f) a list of 3 numbers
answer_12_5g = ...  # g) a list of 3 numbers

for letter, answer in zip("abcdefg", [answer_12_5a, answer_12_5b, answer_12_5c, answer_12_5d, answer_12_5e, answer_12_5f, answer_12_5g]):
    wb.check(f"12.5{letter}", answer)

**Ex 12.7 — PCA à la main en 2D : covariance, axe principal, projection**

In [ ]:
# Ex 12.7: the values you found on paper
answer_12_7a = ...  # a) a list of 2 numbers
answer_12_7b = ...  # b) a list of 2 rows, e.g. [[1, 1], [1, 1]]
answer_12_7c = ...  # c) a list of 2 numbers
answer_12_7d = ...  # d) a number
answer_12_7e = ...  # e) a number
answer_12_7f = ...  # f) a number with 3 decimals
answer_12_7g = ...  # g) a list of 5 numbers
answer_12_7h = ...  # h) a number
answer_12_7i = ...  # i) a list of 2 numbers
answer_12_7j = ...  # j) a number with 3 decimals
answer_12_7k = ...  # k) a number with 3 decimals
answer_12_7l = ...  # l) a number
answer_12_7m = ...  # m) True or False

for letter, answer in zip("abcdefghijklm", [answer_12_7a, answer_12_7b, answer_12_7c, answer_12_7d, answer_12_7e, answer_12_7f, answer_12_7g, answer_12_7h, answer_12_7i, answer_12_7j, answer_12_7k, answer_12_7l, answer_12_7m]):
    wb.check(f"12.7{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire, pour chaque étape d'une préparation, ce qu'elle apprend des données, et donc où faire son `fit` ?
2. Sais-tu choisir un encodage et une mise à l'échelle selon le type des données et le modèle qui suit ?
3. Sais-tu mener une PCA à la main en dimension 2, et dire ce qu'elle garde et ce qu'elle perd ?

**Pour aller plus loin** : le guide « Preprocessing data » et la page « Common pitfalls » de scikit-learn, et le tutoriel de J. Shlens sur la PCA, cités dans la fiche. La suite : le ch. 13 (classifieurs), où les k plus proches voisins et les SVM dépendront de tes mises à l'échelle, et pas les arbres de décision.

*Les exercices de code 12.11 à 12.33 (parties A à D, ta librairie `mylearn.preprocessing`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 12` pour obtenir le notebook complet.*